**Problem Statement**

The Dakota Furniture Company manufactures desks, tables, and chairs. Each furniture item requires lumber, finishing labor, and carpentry labor according to the following specifications:


* **Desk:** 8 board feet of lumber, 4 finishing hours, 2 carpentry hours; sells for $60.

* **Table:** 6 board feet of lumber, 2 finishing hours, 1.5 carpentry hours; sells for $30.

* **Chair:** 1 board foot of lumber, 1.5 finishing hours, 0.5 carpentry hours; sells for $20.

The total available resources are 48 board feet of lumber, 20 finishing hours, and 8 carpentry hours. Demand for desks and chairs is unlimited, while at most 5 tables can be sold. Dakota aims to determine the production quantity for each item to maximize total revenue.

---

**Mathematical Formulation**

**Decision Variables**

* $x_1$: Number of desks produced
* $x_2$: Number of tables produced
* $x_3$: Number of chairs produced

---

**Objective Function**

Maximize total revenue ($Z$):

$$\max Z = 60x_1 + 30x_2 + 20x_3$$

---

**Constraints**

* **Lumber Availability (board feet):**
$$8x_1 + 6x_2 + x_3 \le 48$$


* **Finishing Labor Availability (hours):**
$$4x_1 + 2x_2 + 1.5x_3 \le 20$$


* **Carpentry Labor Availability (hours):**
$$2x_1 + 1.5x_2 + 0.5x_3 \le 8$$


* **Table Demand Upper Bound:**
$$x_2 \le 5$$


* **Non-negativity Constraints:**
$$x_1 \ge 0, \quad x_2 \ge 0, \quad x_3 \ge 0$$



---

In [1]:
!pip install pyomo
!apt-get install -y -qq glpk-utils

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 118337 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b

In [2]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

In [3]:
model = pyo.ConcreteModel()

model.x1 = pyo.Var(domain=pyo.NonNegativeReals)
model.x2 = pyo.Var(domain=pyo.NonNegativeReals)
model.x3 = pyo.Var(domain=pyo.NonNegativeReals)

model.obj = pyo.Objective(expr=60*model.x1 + 30*model.x2 + 20*model.x3, sense=pyo.maximize)

model.const1 = pyo.Constraint(expr=8*model.x1 + 6*model.x2 + model.x3 <= 48)
model.const2 = pyo.Constraint(expr=4*model.x1 + 2*model.x2 + 1.5*model.x3 <= 20)
model.const3 = pyo.Constraint(expr=2*model.x1 + 1.5*model.x2 + 0.5*model.x3 <= 8)
model.const4 = pyo.Constraint(expr=model.x2 <= 5)

optm = SolverFactory('glpk')
results = optm.solve(model)

print(f"Status: {results.solver.status}")
print(f"Optimal Revenue: ${pyo.value(model.obj):.2f}\n")
print(f"Desks (x1)  : {pyo.value(model.x1)}")
print(f"Tables (x2) : {pyo.value(model.x2)}")
print(f"Chairs (x3) : {pyo.value(model.x3)}")

Status: ok
Optimal Revenue: $280.00

Desks (x1)  : 2.0
Tables (x2) : 0.0
Chairs (x3) : 8.0


When can also solve the probelm using sets and parameters in pyomo, when the number variables increases.

In [6]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

In [26]:
from IPython.core.application import crashhandler
model = pyo.ConcreteModel()

# sets
model.i = pyo.Set(initialize=['Desk', 'Table', 'Chair'])

# for i in model.i:
#   print(i)

# parameters
model.lumber = pyo.Param(model.i, initialize={'Desk': 8, 'Table': 6, 'Chair': 1})
lumber = model.lumber
model.finishing = pyo.Param(model.i, initialize={'Desk': 4, 'Table': 2, 'Chair': 1.5})
finishing = model.finishing
model.carpentry = pyo.Param(model.i, initialize={'Desk': 2, 'Table': 1.5, 'Chair': 0.5})
carpentry = model.carpentry
model.price = pyo.Param(model.i, initialize={'Desk': 60, 'Table': 30, 'Chair': 20})
price = model.price

# for i in price:
#   print(i, price[i])

# Decision Variables
model.x = pyo.Var(model.i, domain=pyo.NonNegativeReals)
# for i in model.x:
#     print(i, model.x[i])

# Objective function
def max_profit(model):
  return sum(price[i] * model.x[i] for i in model.i)

model.obj = pyo.Objective(rule=max_profit, sense=pyo.maximize)

# Constraints
def Constraint1(model,i):
  return sum(lumber[i]*model.x[i] for i in model.i)<=48
model.Const1 = pyo.Constraint(model.i,rule=Constraint1)

def Constraint2(model,i):
  return sum(finishing[i]*model.x[i] for i in model.i)<=20
model.Const2 = pyo.Constraint(model.i,rule=Constraint2)

def Constraint3(model,i):
  return sum(carpentry[i]*model.x[i] for i in model.i)<=8
model.Const3 = pyo.Constraint(model.i,rule=Constraint3)

def Constraint4(model,i):
  if i == 'Table':
    return model.x[i]<=5
  else:
    return pyo.Constraint.Skip
model.Const4 = pyo.Constraint(model.i,rule=Constraint4)

# Solve
solver = SolverFactory('glpk')
results = solver.solve(model)

print(f"Status: {results.solver.status}")
print(f"Optimal Revenue: ${pyo.value(model.obj):.2f}\n")

print(f"Desks (x1)  : {pyo.value(model.x['Desk'])}")
print(f"Tables (x2) : {pyo.value(model.x['Table'])}")
print(f"Chairs (x3) : {pyo.value(model.x['Chair'])}")

Status: ok
Optimal Revenue: $280.00

Desks (x1)  : 2.0
Tables (x2) : 0.0
Chairs (x3) : 8.0
